# QKD Information Reconciliation — Benchmark on Google Colab

This notebook runs the Phase 1 benchmark for the `qkd-cascade-ldpc` project
on **Colab's CPU runtime**. No GPU needed (Python loops + discrete-event
simulation; GPU gives no speedup).

Default config: n=8192, 200 frames/Q → **~40-60 min wall-clock** (serial,
because multiprocessing in Colab notebooks is unreliable).

**Workflow:**
1. Cell 1 installs deps and auto-restarts the kernel. Wait for restart, then SKIP Cell 1.
2. Cells 2-4: setup (~30s).
3. Cells 5-6: the actual sweep (~50 min).
4. Cells 7-9: plots + zip download.

## Cell 1 — Install dependencies (auto-restarts kernel)

In [ ]:
import os, sys

SENTINEL = "/content/.deps_installed"

if not os.path.exists(SENTINEL):
    print("Installing scientific stack (pinned for consistency)...")
    # --no-deps avoids pip's dependency resolver hanging on transitive deps
    !pip install --force-reinstall --no-deps "numpy<2.1" "scipy<1.14" "pandas<3.0" pyarrow
    print("\nInstalling sequence library...")
    !pip install --quiet sequence
    open(SENTINEL, "w").write("done")
    print("\n" + "=" * 60)
    print("DONE installing. Restarting kernel automatically...")
    print("After restart, SKIP this cell and run from Cell 2.")
    print("=" * 60)
    os._exit(0)   # Triggers Colab kernel restart
else:
    print("Deps already installed (sentinel file present); skipping.")

## Cell 2 — Verify package versions and clone the repo

If you see a `numpy` import error here, the kernel didn't restart — go
back to Cell 1 and run it (it'll trigger another restart).

In [ ]:
import numpy, scipy, pandas, pyarrow
print(f"numpy  : {numpy.__version__}  (expected 2.0.x)")
print(f"scipy  : {scipy.__version__}  (expected 1.13.x or earlier)")
print(f"pandas : {pandas.__version__}  (expected 2.x)")
print(f"pyarrow: {pyarrow.__version__}")

import os, sys, subprocess

REPO_URL = "https://github.com/alexandrachirita98/qkd-cascade-ldpc.git"
REPO_DIR = "/content/qkd-cascade-ldpc"

if not os.path.exists(REPO_DIR):
    print(f"\nCloning {REPO_URL}...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    print(f"\nRepo already at {REPO_DIR}; pulling latest...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f"\ncwd        : {os.getcwd()}")
print(f"sys.path[0]: {sys.path[0]}")

## Cell 3 — Verify the LDPC code pool

In [ ]:
from src.codes.storage import list_available

pool = list_available()
print(f"Available codes: {len(pool)}\n")
for n, r in pool:
    print(f"  n={n:>5}  R={r:.2f}")

n8192 = sum(1 for n, _ in pool if n == 8192)
if n8192 < 9:
    print(f"\n[!] only {n8192}/9 n=8192 codes; generating (~5 min)...")
    !python -m src.tools.generate_codes --frame-lengths 8192
    pool = list_available()
    print(f"\nAfter: {sum(1 for n,_ in pool if n==8192)} n=8192 codes")

## Cell 4 — Sweep parameters

Adjust to your taste. The estimate at the end tells you roughly how long
the whole thing will take.

In [ ]:
N             = 8192     # 1024 (fast, ~10 min) or 8192 (canonical, ~50 min)
ALPHA         = 0.15
FRAMES_PER_Q  = 200      # 1000 = paper-grade; 200 is a good compromise on Colab
SEED          = 42

payload = N - round(ALPHA * N)
QBERS = [round(0.005 * (i + 1), 4) for i in range(20)]   # 0.005 .. 0.100

# Rough per-frame estimate (varies with Q and algorithm)
per_frame_s = 0.20 if N >= 8192 else 0.04
n_frames_total = len(QBERS) * FRAMES_PER_Q * 3
est_min = n_frames_total * per_frame_s / 60

print(f"N            = {N}")
print(f"alpha        = {ALPHA}")
print(f"payload/frame= {payload}")
print(f"frames/Q     = {FRAMES_PER_Q}")
print(f"QBER points  = {len(QBERS)} from {QBERS[0]} to {QBERS[-1]}")
print(f"Total frames : {n_frames_total} (across all 3 algorithms)")
print(f"\nEstimated wall-clock (serial): ~{est_min:.0f} min")

## Cell 5 — QBER sweep (SERIAL — no multiprocessing)

Serial because Colab notebooks hang on `multiprocessing.Pool` — kernel
state doesn't fork cleanly. Each Q point's progress is printed live so
you can see it advancing.

Saves to `/content/qber_sweep.parquet`. Re-running overwrites.

In [ ]:
import pandas as pd
import time
from src.harness import make_cascade, make_mueller, make_borisov, generate_frames

t0 = time.perf_counter()
parts = []
print(f"QBER sweep: {len(QBERS)} points (serial, ~{est_min:.0f} min total)...\n")

for i, q in enumerate(QBERS):
    t_q = time.perf_counter()
    print(f"  [{i+1:>2}/{len(QBERS)}] Q={q:.3f} ... ", end="", flush=True)

    algos = [
        make_cascade(seed=SEED),
        make_mueller(n=N, seed=SEED),
        make_borisov(n=N, alpha=ALPHA, seed=SEED),
    ]
    frames = generate_frames(q, payload, FRAMES_PER_Q, seed=SEED)
    rows = []
    for alg in algos:
        for j, (a, b, true_q) in enumerate(frames):
            res = alg.fn(a, b, q, true_q)
            rows.append({
                "q": q, "alg": alg.name, "frame_idx": j,
                "leakage_bits": res.leakage_bits, "messages": res.messages,
                "iterations": res.iterations, "success": res.success,
                "wall_clock_s": res.wall_clock_s, "true_qber": res.true_qber,
            })
    parts.append(pd.DataFrame(rows))
    print(f"done in {time.perf_counter()-t_q:>5.0f}s  "
          f"(elapsed {(time.perf_counter()-t0)/60:>5.1f}m)")

df_q = pd.concat(parts, ignore_index=True)
df_q.to_parquet("/content/qber_sweep.parquet", index=False)
print(f"\nQBER sweep total: {(time.perf_counter()-t0)/60:.1f}m  |  "
      f"{len(df_q)} records → /content/qber_sweep.parquet")

## Cell 6 — Mismatch sweep (also serial)

In [ ]:
from src.harness import run_mismatch_sweep, make_cascade, make_mueller, make_borisov

algorithms = [
    make_cascade(seed=SEED),
    make_mueller(n=N, seed=SEED),
    make_borisov(n=N, alpha=ALPHA, seed=SEED),
]

print(f"Mismatch sweep (3 trueQ x 5 deltas x {FRAMES_PER_Q} frames x 3 alg)...\n")
t0 = time.perf_counter()
df_m = run_mismatch_sweep(
    algorithms,
    true_qbers=[0.02, 0.04, 0.06],
    deltas=[-0.02, -0.01, 0.0, 0.01, 0.02],
    n_frames_per_point=FRAMES_PER_Q,
    n_payload=payload,
    seed=SEED,
    progress_callback=lambda m: print(f"  {m}"),
)
df_m.to_parquet("/content/mismatch_sweep.parquet", index=False)
print(f"\nMismatch sweep total: {(time.perf_counter()-t0)/60:.1f}m  |  "
      f"{len(df_m)} records → /content/mismatch_sweep.parquet")

## Cell 7 — Render and display the 7 slide-deck plots

In [ ]:
from src.harness import make_slide_deck
from pathlib import Path
from IPython.display import Image, display, Markdown

out_dir = Path("/content/plots")
paths = make_slide_deck(df_q, df_m, n_payload=payload, out_dir=out_dir)
print(f"rendered {len(paths)} plots → {out_dir}/\n")

for name, p in paths.items():
    display(Markdown(f"### `{name}`"))
    display(Image(str(p)))

## Cell 8 — Summary table

In [ ]:
from src.harness import per_qa_summary

summary = per_qa_summary(df_q, n_payload=payload)
cols = ["alg", "q", "FER", "f", "f_eff", "mean_messages", "mean_wall_ms", "R_sec_per_block"]
display(summary[cols].round(4))

## Cell 9 — Bundle and download

Triggers a browser download of all artifacts (parquets + plots). On
Colab, files in `/content/` are lost when the session ends, so download
this zip before disconnecting.

In [ ]:
import shutil, os

# Stage everything for the zip
stage = "/content/results_staging"
shutil.rmtree(stage, ignore_errors=True)
os.makedirs(stage, exist_ok=True)
shutil.copy("/content/qber_sweep.parquet", stage)
shutil.copy("/content/mismatch_sweep.parquet", stage)
shutil.copytree("/content/plots", os.path.join(stage, "plots"))

zip_path = shutil.make_archive("/content/qkd_sweep_results", "zip", stage)
size_mb = os.path.getsize(zip_path) / 1024 / 1024
print(f"Created {zip_path} ({size_mb:.1f} MB)")

# Colab-native download (triggers browser save dialog)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("(Not in Colab — manually grab the file from the Files panel)")